# Regional Feature Engineering

This notebook:

1. loads processed weekly storage and weekly weather for an EIA region;
2. joins storage and weather on week-ending Friday;
3. engineers model-ready features (calendar, weather lags, storage lags);
4. validates and exports the feature table for modeling notebooks.

Change `REGION` to rebuild features for Lower 48 or any of the five EIA storage regions.

For a routine refresh without re-running cells here, use:

```bash
gas-data refresh --region R48 --only features
```

In [1]:
from pathlib import Path

import pandas as pd

from gas_forecast.data.export import save_versioned_parquet
from gas_forecast.data.features import (
    DEFAULT_WEATHER_MODEL_FEATURES,
    TARGET_COLUMN,
    build_weekly_model_features,
    validate_weekly_model_features,
)
from gas_forecast.data.paths import latest_processed_path
from gas_forecast.data.regions import region_slug, supported_storage_regions

In [2]:
REGION = "R48"  # R48, R01, R02, R03, R04, R05
REGION_SLUG = region_slug(REGION)
PROCESSED_DIR = Path("../datasets/processed")

STORAGE_PATH = latest_processed_path(REGION, "weekly_storage", PROCESSED_DIR)
WEEKLY_WEATHER_PATH = latest_processed_path(REGION, "weekly_weather", PROCESSED_DIR)
FEATURES_PATH = latest_processed_path(REGION, "weekly_model_features", PROCESSED_DIR)

In [3]:
storage = pd.read_parquet(STORAGE_PATH)
weekly_weather = pd.read_parquet(WEEKLY_WEATHER_PATH)

assert (storage["duoarea"] == REGION).all(), (
    f"Storage duoarea mismatch for {REGION}"
)
assert (weekly_weather["duoarea"] == REGION).all(), (
    f"Weather duoarea mismatch for {REGION}"
)

print(f"Storage rows: {len(storage):,}")
print(f"Weather rows: {len(weekly_weather):,}")

Storage rows: 859
Weather rows: 858


In [4]:
weekly_model_features = build_weekly_model_features(
    storage,
    weekly_weather,
    region=REGION,
)

validate_weekly_model_features(weekly_model_features, region=REGION)

print(f"Target: {TARGET_COLUMN}")
print(f"Default weather model features: {DEFAULT_WEATHER_MODEL_FEATURES}")
weekly_model_features.head()

Target: weekly_change_bcf
Default weather model features: ('week_sin', 'week_cos', 'month_sin', 'month_cos', 'hdd', 'cdd', 'hdd_lag1', 'weekly_change_lag1')


,date,storage_bcf,weekly_change_bcf,year,month,week_of_year,duoarea,temperature_f,hdd,cdd,...,hdd_per_day,cdd_per_day,hdd_lag1,cdd_lag1,hdd_lag4,cdd_lag4,weekly_change_lag1,weekly_change_lag4,storage_bcf_lag52,weekly_change_yoy
0,2010-01-15,2607,-243.0,2010,1,2,R48,32.524329,227.329697,0.000000,...,32.475671,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-22,2521,-86.0,2010,1,3,R48,42.249004,160.262816,1.005842,...,22.894688,0.143692,227.329697,0.000000,NaN,NaN,-243.0,NaN,NaN,NaN
2,2010-01-29,2406,-115.0,2010,1,4,R48,38.978161,182.663103,0.510228,...,26.094729,0.072890,160.262816,1.005842,NaN,NaN,-86.0,NaN,NaN,NaN
3,2010-02-05,2214,-192.0,2010,2,5,R48,35.025938,210.152041,0.333610,...,30.021720,0.047659,182.663103,0.510228,NaN,NaN,-115.0,NaN,NaN,NaN
4,2010-02-12,2026,-188.0,2010,2,6,R48,32.707883,226.044819,0.000000,...,32.292117,0.000000,210.152041,0.333610,227.329697,0.0,-192.0,-243.0,NaN,NaN


In [5]:
save_versioned_parquet(
    weekly_model_features,
    PROCESSED_DIR,
    f"{REGION_SLUG}_weekly_model_features",
    save_latest=True,
)

WindowsPath('../datasets/processed/lower48_weekly_model_features_20260630T155610Z.parquet')

## Optional: build all regions

Uncomment and run after `01a` and `01b` have been executed for each region.

In [6]:
# from gas_forecast.pipelines import run_data_pipeline
#
# for region in supported_storage_regions():
#     run_data_pipeline(region, stages=("features",))